# Replication Starter: Lee (2008), Incumbency Advantage in US House Elections
## ECON 5200 · Midterm Replication Study · regression discontinuity

Lee, D. S. (2008). "Randomized Experiments from Non-random Selection in U.S. House Elections." *Journal of
Econometrics* 142(2): 675–697.

> **AI rule for this project.** Sections 1 and 2 (positioning, direct replication) are **no AI**: not for
> the code, not for the prose. Sections 3 to 6 allow AI as a documented co-pilot under P.R.I.M.E., with the full
> transcript in `ai-appendix.md`. Every cell below is marked with the section it belongs to.

**How to use this notebook.** It is a starter, not the project. Copy it into your repository made from
[RyanPiao/econ5200-replication-template](https://github.com/RyanPiao/econ5200-replication-template), run it top to
bottom, and fill the cells marked **YOUR TURN**. Cells marked **GIVEN** run as they are; cells marked **CHECK** test
what you wrote and print a message, never the answer. A blank copy runs from top to bottom and prints no estimate:
the numbers appear once you have written Section 2.

In [ ]:
# GIVEN — setup: the two RD packages Colab does not have, then the libraries
%pip install rdrobust==2.0.0 rddensity==3.0 -q

import numpy as np
import pandas as pd
import statsmodels.api as sm
from rdrobust import rdrobust
from rddensity import rddensity

print("Setup complete.")

## The data, and a gap you must document

`lmb-data.dta` is the file of **Lee, Moretti and Butler (2004)**, the same House elections used for a different
question: 13,588 district-years, **1948–1990**. Lee (2008) uses elections from 1946 to 1998. You therefore cannot
expect to match the paper's printed number exactly, and Section 2's table must say so and estimate what the
missing years could do. That is a discrepancy with an investigated cause, which the rubric rewards.

| variable | what it is |
|---|---|
| `demvoteshare` | the Democrat's share of the two-party vote in this election, 0 to 1 (the outcome) |
| `lagdemvoteshare` | the Democrat's share in the previous election in the district, 0 to 1 (the running variable's source) |
| `democrat`, `lagdemocrat` | 1 if the Democrat won this election, the previous one |
| `year`, `state`, `district` | when and where |

The running variable is the previous margin, `lagdemvoteshare − 0.5`: positive means the Democrat won last time.

In [ ]:
# GIVEN — Section 2 starts here: load the data and build the running variable
lmb = pd.read_stata("https://raw.githubusercontent.com/scunning1975/mixtape/master/lmb-data.dta")
lmb = lmb.dropna(subset=["demvoteshare", "lagdemvoteshare"]).copy()
lmb["margin"] = lmb["lagdemvoteshare"] - 0.5
CUTOFF = 0.0
print(f"{len(lmb):,} district-years, {lmb['year'].min():.0f}-{lmb['year'].max():.0f}")
lmb[["demvoteshare", "lagdemvoteshare", "margin", "democrat", "lagdemocrat"]].describe().round(3)

## Section 2: direct replication — YOUR TURN, no AI

Estimate the incumbency advantage the way the paper's headline does: the jump in `demvoteshare` at `margin = 0`,
from a regression on each side of the cutoff, with the paper's choices of bandwidth, functional form and kernel.
Write it yourself with `statsmodels`, not with `rdrobust`.

Set these names, because Section 3 reuses them:

- `H`: your bandwidth on `margin`; `KERNEL`: `"rectangular"` or `"triangular"`; `ORDER`: the polynomial order on each side
- `rd_fit`: your fitted statsmodels result; `JUMP`: your estimate of the jump, as a float

In [ ]:
# YOUR TURN — Section 2, no AI: set H, KERNEL, ORDER, fit rd_fit, and set JUMP
H = None
KERNEL = None
ORDER = None
rd_fit = None
JUMP = None

### Your number, with its unit

`demvoteshare` is a share between 0 and 1, so the jump is a change in a share: read it in **percentage points of
the two-party vote**. A jump of 0.05 is 5 percentage points, not "5 percent". This check accepts only percentage
points. Fill all five names:

- `PRINTED`: the jump exactly as your code prints it
- `IN_UNITS`: the same jump in percentage points
- `UNIT`: `percentage points`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [ ]:
# YOUR TURN — Section 2, no AI
PRINTED = None
IN_UNITS = None
UNIT = None
PAPER_NUMBER = None
PAPER_WHERE = None

In [ ]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

LEE_UNITS = {"percentage points": [lambda b: 100 * b]}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, LEE_UNITS)

## Section 3: robustness — GIVEN code, AI allowed from here

These checks use `rdrobust` and `rddensity`, which the paper predates: a data-driven bandwidth and robust
bias-corrected inference. They take your Section 2 choices. For each, write what it implies about the original
claim.

**1. The data-driven bandwidth, and yours.** The MSE-optimal bandwidth beside the jump at H/2, H and 2H.

In [ ]:
# GIVEN — Section 3: rdrobust at the cutoff, for any bandwidth, kernel, order, outcome and sample
def rd(h=None, kernel="triangular", p=1, outcome="demvoteshare", data=None, cutoff=0.0):
    data = lmb if data is None else data
    kw = {"h": h} if h else {}
    r = rdrobust(y=data[outcome], x=data["margin"], c=cutoff, p=p, kernel=kernel, **kw)
    return {"jump": r.coef.iloc[0, 0], "se": r.se.iloc[0, 0], "p_robust": r.pv.iloc[2, 0],
            "h": r.bws.iloc[0, 0], "n": int(r.N_h[0] + r.N_h[1])}


def show(label, r):
    print(f"{label:<34} {r['jump']:+.4f} (SE {r['se']:.4f}, robust p {r['p_robust']:.3f}), h = {r['h']:.3f}, n = {r['n']:,}")


if H is None:
    print("Fill Section 2 first: H is not set yet.")
else:
    k = "uniform" if KERNEL == "rectangular" else "triangular"
    show("MSE-optimal h", rd())
    for mult in (0.5, 1, 2):
        show(f"h = {mult:g} x {H}", rd(h=mult * H, kernel=k, p=ORDER))

**2. Functional form, manipulation, and placebo cutoffs.** A local quadratic beside your order; the density test,
which asks whether close races pile up on the winning side (a question with its own literature for the House:
see Caughey and Sekhon 2011); and placebo cutoffs inside each side, where nothing should jump.

In [ ]:
# GIVEN — Section 3: order 2, the density test, and placebo cutoffs at -0.1 and +0.1
if H is None:
    print("Fill Section 2 first: H is not set yet.")
else:
    show("local quadratic, MSE-optimal h", rd(p=2))
    dens = rddensity(X=lmb["margin"].to_numpy(), c=0.0)
    print(f"Density test at 0: t = {dens.test['t_jk']:+.3f}, p = {dens.test['p_jk']:.3f}")
    show("placebo cutoff -0.1 (losers only)", rd(data=lmb[lmb["margin"] < 0], cutoff=-0.1))
    show("placebo cutoff +0.1 (winners only)", rd(data=lmb[lmb["margin"] > 0], cutoff=0.1))

**3. Your own specification.** Change at least one choice and justify it: the other outcome (`democrat`, the
probability of winning, which Lee also reports), a period (before or after a redistricting cycle, or by decade), a
kernel, an order, or a bandwidth. `MY_SPEC` is a dictionary: `{"outcome": "demvoteshare" or "democrat", "h": a
bandwidth or None for MSE-optimal, "kernel": "uniform" or "triangular", "p": 1 or 2, "sample": None or a pandas query
string}`.

In [ ]:
# YOUR TURN — Section 3: your specification, and two sentences on why it is defensible
MY_SPEC = None
MY_SPEC_WHY = None

In [ ]:
# CHECK — Section 3: your own specification. Run it; you need not read it.
def check_my_spec(my_spec, baseline, why):
    if my_spec is None or why is None:
        print("Not filled yet: set MY_SPEC and MY_SPEC_WHY above.")
        return False
    if my_spec == baseline:
        print("MY_SPEC is the baseline. Change at least one choice, and say why that choice is defensible.")
        return False
    sentences = [s for s in str(why).replace("?", ".").split(".") if len(s.split()) >= 5]
    if len(sentences) < 2:
        print("MY_SPEC_WHY needs at least two sentences: what you changed, and why a careful author might have made that choice.")
        return False
    print("Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.")
    return True

BASELINE = None if H is None else {"outcome": "demvoteshare", "h": H,
                                   "kernel": "uniform" if KERNEL == "rectangular" else "triangular", "p": ORDER, "sample": None}
if check_my_spec(MY_SPEC, BASELINE, MY_SPEC_WHY):
    d = lmb.query(MY_SPEC["sample"]) if MY_SPEC["sample"] else lmb
    show("your specification", rd(h=MY_SPEC["h"], kernel=MY_SPEC["kernel"], p=MY_SPEC["p"], outcome=MY_SPEC["outcome"], data=d))
    print(f"Section 2 baseline: {JUMP:+.4f}")

**4. Your other exercises.** Section 3 asks for at least four. Natural ones here: the jump by decade; the
probability of winning beside the vote share; covariates that should not jump (the previous election's
characteristics); and what the 1990–1998 elections missing from this file could do to the estimate. Write them
below this cell.

## Sections 4 to 6, in your repository

- **Section 4, does the finding hold?** Use your Section 3 results. "Holds in a narrower population", "holds at a
  smaller magnitude" and "holds under an assumption the authors did not state" are all answers.
- **Section 5, peer audit.** Assigned on the date the brief gives, never on your own paper; file the issue by the
  project deadline in the brief's Checkpoints list.
- **Section 6, your own repository.** Move the code you keep from this notebook into the template's layout, pin
  versions in `requirements.txt` (this notebook was built with pandas 2.2.2, numpy 2.0.2, statsmodels 0.14.6,
  linearmodels 7.1, rdrobust 2.0.0 and rddensity 3.0), and make the README name the one command that reproduces
  your headline table. Then clone your own repository into an empty folder and run that command.